# Integración de Datos - Calidad del Aire y Salud
**Materia:** Integración y Adquisición de Datos

In [ ]:
import pandas as pd
import psycopg2
from psycopg2.extras import RealDictCursor
import os
from dotenv import load_dotenv

load_dotenv('../backend/.env')
DATABASE_URL = os.getenv('DATABASE_URL')

conn = psycopg2.connect(DATABASE_URL, cursor_factory=RealDictCursor)
print('Conectado a PostgreSQL')

## 1. Datos disponibles por tabla

In [ ]:
query_conteo = '''
SELECT 'municipios' AS tabla, COUNT(*) AS total FROM municipios
UNION ALL SELECT 'contaminantes', COUNT(*) FROM contaminantes
UNION ALL SELECT 'grupos_diagnosticos', COUNT(*) FROM grupos_diagnosticos
UNION ALL SELECT 'analisis_riesgo', COUNT(*) FROM analisis_riesgo
UNION ALL SELECT 'enfermedades', COUNT(*) FROM enfermedades
UNION ALL SELECT 'tasa_mortalidad', COUNT(*) FROM tasa_mortalidad
UNION ALL SELECT 'defunciones', COUNT(*) FROM defunciones
UNION ALL SELECT 'mediciones_siata', COUNT(*) FROM mediciones_siata;
'''
pd.read_sql(query_conteo, conn)

## 2. Integración con JOINs: crear tabla consolidada

In [ ]:
query_integracion = '''
DROP TABLE IF EXISTS consolidado_salud;

CREATE TABLE consolidado_salud AS
WITH ultima_siata AS (
    SELECT DISTINCT ON (estacion) 
        estacion, pm25_ica, categoria_ica, fecha_hora
    FROM mediciones_siata
    WHERE pm25_ica IS NOT NULL
    ORDER BY estacion, fecha_hora DESC
)
SELECT 
    m.id AS municipio_id,
    m.nombre_normalizado AS municipio,
    us.estacion AS estacion_siata,
    us.pm25_ica AS pm25_actual,
    us.categoria_ica,
    us.fecha_hora AS ultima_medicion,
    gd.id AS grupo_diagnostico_id,
    gd.nombre AS enfermedad,
    gd.codigo_cie10,
    ar.sexo,
    ar.grupo_edad,
    ar.beta AS coeficiente_riesgo,
    ar.porcentaje_incremento_riesgo AS incremento_basico_historico,
    ROUND((EXP(ar.beta * us.pm25_ica) - 1) * 100, 2) AS incremento_riesgo_pct_actual
FROM ultima_siata us
JOIN municipios m ON us.estacion ILIKE '%' || m.nombre_normalizado || '%'
JOIN analisis_riesgo ar ON ar.municipio_id = m.id
JOIN grupos_diagnosticos gd ON ar.grupo_diagnostico_id = gd.id
JOIN contaminantes c ON ar.contaminante_id = c.id
WHERE c.nombre = 'pm25';
'''

cur = conn.cursor()
cur.execute(query_integracion)
conn.commit()
print('Tabla consolidado_salud creada con JOINs entre SIATA + riesgo + municipios + enfermedades')

## 3. Verificar datos integrados

In [ ]:
df = pd.read_sql('SELECT * FROM consolidado_salud ORDER BY incremento_riesgo_pct_actual DESC;', conn)
print(f'Registros: {len(df)}, Columnas: {list(df.columns)}')
df.head(10)

## 4. Exportar a CSV

In [ ]:
df.to_csv('datos_integrados.csv', index=False, sep=';')
print(f'Exportado: datos_integrados.csv ({len(df)} filas)')

In [ ]:
cur.close()
conn.close()
print('Conexión cerrada.')